In [3]:
### Import Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LambdaCallback
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage.transform import resize
from sklearn.model_selection import train_test_split

### Data Paths
# Original data
train_mask_folder = "Train-Mask1"
train_input_folder = "Train-Input1"
test_input_folder = "Test-Input"
test_mask_folder = "Test-Mask"

# Combined data (1500 total)
combined_input_folder = "final_generated_input"  # Adjust if different
combined_mask_folder = "final_generated_mask"    # Adjust if different

def dice_acc(y_true, y_pred, smooth=1e-7):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    union = tf.reduce_sum(y_true + y_pred, axis=[1, 2, 3])
    return tf.reduce_mean((2. * intersection + smooth) / (union + smooth))

def dice_loss(y_true, y_pred):
    return 1 - dice_acc(y_true, y_pred)

# Custom Metrics
def precision(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    true_positives = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    predicted_positives = tf.reduce_sum(y_pred, axis=[1, 2, 3])
    return tf.reduce_mean(true_positives / (predicted_positives + tf.keras.backend.epsilon()))

def recall(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    true_positives = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    actual_positives = tf.reduce_sum(y_true, axis=[1, 2, 3])
    return tf.reduce_mean(true_positives / (actual_positives + tf.keras.backend.epsilon()))

def accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    return tf.reduce_mean(tf.keras.metrics.binary_accuracy(y_true, y_pred))

### U-Net Model Definition (from Segmentation_GPU.ipynb)
def unet(input_shape=(256, 256, 1)):
    inputs = layers.Input(input_shape)
    
    # Encoder
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    
    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)
    
    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c4)
    p4 = layers.MaxPooling2D((2, 2))(c4)
    
    # Bottleneck
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(p4)
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(c5)
    
    # Decoder
    u6 = layers.Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = layers.concatenate([u6, c4])
    c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(u6)
    c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c6)
    
    u7 = layers.Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = layers.concatenate([u7, c3])
    c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(u7)
    c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c7)
    
    u8 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c7)
    u8 = layers.concatenate([u8, c2])
    c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u8)
    c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c8)
    
    u9 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c8)
    u9 = layers.concatenate([u9, c1])
    c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u9)
    c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c9)
    
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c9)
    
    model = Model(inputs=[inputs], outputs=[outputs])
    return model

def load_images(folder, is_mask=False):
    images = []
    valid_filenames = []
    all_filenames = os.listdir(folder)
    print(f"Total files in {folder}: {len(all_filenames)}")
    for filename in all_filenames:
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(folder, filename)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = resize(img, (256, 256), mode='constant', preserve_range=True)
                if is_mask and np.max(img) > 0:
                    img = (img > 127).astype(np.float32)
                images.append(img)
                valid_filenames.append(filename)
            else:
                print(f"Failed to load {img_path}")
    print(f"Loaded {len(images)} valid images from {folder}: {valid_filenames[:5]}... (total valid files: {len(valid_filenames)})")
    return np.array(images)

# Load original data for baseline and validation split
print("Loading Original Train Inputs...")
Train_Inputs = load_images(train_input_folder)
print("Loading Original Train Masks...")
Train_Masks = load_images(train_mask_folder, is_mask=True)
print("Loading Test Inputs...")
Test_Inputs = load_images(test_input_folder)
print("Loading Test Masks...")
Test_Masks = load_images(test_mask_folder, is_mask=True)

# Load combined data (1500 total)
print("Loading Combined Inputs...")
Combined_Inputs = load_images(combined_input_folder)
print("Loading Combined Masks...")
Combined_Masks = load_images(combined_mask_folder, is_mask=True)

# Expand dims for channel (grayscale)
Train_Inputs = np.expand_dims(Train_Inputs, axis=-1)
Train_Masks = np.expand_dims(Train_Masks, axis=-1)
Test_Inputs = np.expand_dims(Test_Inputs, axis=-1)
Test_Masks = np.expand_dims(Test_Masks, axis=-1)
Combined_Inputs = np.expand_dims(Combined_Inputs, axis=-1)
Combined_Masks = np.expand_dims(Combined_Masks, axis=-1)

# Check for invalid values
print("Checking for NaN/Inf in Train_Inputs:", np.any(np.isnan(Train_Inputs)), np.any(np.isinf(Train_Inputs)))
print("Checking for NaN/Inf in Train_Masks:", np.any(np.isnan(Train_Masks)), np.any(np.isinf(Train_Masks)))
print("Mask fraction of 1s:", np.mean(Train_Masks), np.mean(Train_Masks > 0))

# Ensure matching sizes for original and combined
if len(Train_Inputs) != len(Train_Masks):
    min_len = min(len(Train_Inputs), len(Train_Masks))
    Train_Inputs = Train_Inputs[:min_len]
    Train_Masks = Train_Masks[:min_len]
    print(f"Trimmed original data to match: {min_len} samples")

if len(Combined_Inputs) != len(Combined_Masks):
    min_len = min(len(Combined_Inputs), len(Combined_Masks))
    Combined_Inputs = Combined_Inputs[:min_len]
    Combined_Masks = Combined_Masks[:min_len]
    print(f"Trimmed combined data to match: {min_len} samples")

if len(Train_Inputs) != len(Train_Masks):
    raise ValueError(f"Mismatch in original data after trimming: {len(Train_Inputs)} inputs vs {len(Train_Masks)} masks")
if len(Combined_Inputs) != len(Combined_Masks):
    raise ValueError(f"Mismatch in combined data after trimming: {len(Combined_Inputs)} inputs vs {len(Combined_Masks)} masks")

print(f"Original Training Inputs: {Train_Inputs.shape}, Masks: {Train_Masks.shape}")
print(f"Combined Inputs: {Combined_Inputs.shape}, Masks: {Combined_Masks.shape}")

# Split original training data to get 100 validation samples (reused for both trainings)
if len(Train_Inputs) >= 100:
    Train_Inputs, Val_Inputs, Train_Masks, Val_Masks = train_test_split(
        Train_Inputs, Train_Masks, test_size=100, random_state=42
    )
    print(f"Validation set created from original training data: {Val_Inputs.shape}, Masks: {Val_Masks.shape}")
    print(f"Remaining Original Training set: {Train_Inputs.shape}, Masks: {Train_Masks.shape}")
else:
    raise ValueError(f"Original training set has only {len(Train_Inputs)} samples, less than 100 for validation.")

### Custom Callback for Epoch Logging
epoch_callback = LambdaCallback(
    on_epoch_end=lambda epoch, logs: print(f"Epoch {epoch + 1}: Loss = {logs['loss']:.4f}, Dice Acc = {logs['dice_acc']:.4f}, Precision = {logs['precision']:.4f}, Recall = {logs['recall']:.4f}, Accuracy = {logs['accuracy']:.4f}, Val Loss = {logs['val_loss']:.4f}, Val Dice Acc = {logs['val_dice_acc']:.4f}, Val Precision = {logs['val_precision']:.4f}, Val Recall = {logs['val_recall']:.4f}, Val Accuracy = {logs['val_accuracy']:.4f}")
)

### Baseline Training (Original Data Only)
print("Training Baseline Model on Original Data...")
model_baseline = unet(input_shape=(256, 256, 1))
model_baseline.compile(optimizer=Adam(learning_rate=1e-4), loss=dice_loss, metrics=[dice_acc, precision, recall, accuracy])

early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

results_baseline = model_baseline.fit(
    Train_Inputs, Train_Masks,
    batch_size=8,
    epochs=100,
    validation_data=(Val_Inputs, Val_Masks),
    callbacks=[early_stop, reduce_lr, epoch_callback],
    verbose=1
)

baseline_val_dice = max(results_baseline.history['val_dice_acc'])
print(f"\nBaseline Max Validation Dice Score: {baseline_val_dice:.4f}")

### Combined Training (1500 Data)
print("\nTraining Model on Combined Data...")
model_combined = unet(input_shape=(256, 256, 1))
model_combined.compile(optimizer=Adam(learning_rate=1e-4), loss=dice_loss, metrics=[dice_acc, precision, recall, accuracy])

results_combined = model_combined.fit(
    Combined_Inputs, Combined_Masks,
    batch_size=8,
    epochs=100,
    validation_data=(Val_Inputs, Val_Masks),  # Reuse the same validation set
    callbacks=[early_stop, reduce_lr, epoch_callback],
    verbose=1
)

combined_val_dice = max(results_combined.history['val_dice_acc'])
print(f"\nCombined Max Validation Dice Score: {combined_val_dice:.4f}")

### Compare Results
print(f"Improvement: {combined_val_dice - baseline_val_dice:.4f} (Baseline: {baseline_val_dice:.4f} -> Combined: {combined_val_dice:.4f})")

### Plot Training History
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(results_baseline.history['dice_acc'], label='Train (Baseline)')
plt.plot(results_baseline.history['val_dice_acc'], label='Val (Baseline)')
plt.title('Baseline Dice-Score')
plt.ylabel('DSC')
plt.xlabel('Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(results_combined.history['dice_acc'], label='Train (Combined)')
plt.plot(results_combined.history['val_dice_acc'], label='Val (Combined)')
plt.title('Combined Dice-Score')
plt.ylabel('DSC')
plt.xlabel('Epochs')
plt.legend()

plt.tight_layout()
plt.show()

### Evaluation and Prediction (Combined Model)
preds_test = model_combined.predict(Test_Inputs, verbose=1)
preds_test_t = (preds_test > 0.5).astype(np.uint8)

### Visualize Random Prediction
ix = np.random.randint(0, len(Test_Inputs)-1)
print(f"\nRandom Index: {ix}")

print('Test Image')
plt.imshow(Test_Inputs[ix].squeeze(), cmap='gray')
plt.show()

print('Predicted Mask')
plt.imshow(preds_test_t[ix].squeeze(), cmap='gray')
plt.show()

print('True Mask')
plt.imshow(Test_Masks[ix].squeeze(), cmap='gray')
plt.show()

# Save models
model_baseline.save('unet_baseline.keras')
model_combined.save('unet_combined.keras')

Loading Original Train Inputs...
Total files in Train-Input1: 685
Loaded 685 valid images from Train-Input1: ['A (1000).jpg', 'A (1001).jpg', 'A (1002).jpg', 'A (1003).jpg', 'A (1004).jpg']... (total valid files: 685)
Loading Original Train Masks...
Total files in Train-Mask1: 685
Loaded 685 valid images from Train-Mask1: ['A (1000).png', 'A (1001).png', 'A (1002).png', 'A (1003).png', 'A (1004).png']... (total valid files: 685)
Loading Test Inputs...
Total files in Test-Input: 53
Loaded 53 valid images from Test-Input: ['A (1351).jpg', 'A (1352).jpg', 'A (1353).jpg', 'A (1354).jpg', 'A (1355).jpg']... (total valid files: 53)
Loading Test Masks...
Total files in Test-Mask: 53
Loaded 53 valid images from Test-Mask: ['A (1351).png', 'A (1352).png', 'A (1353).png', 'A (1354).png', 'A (1355).png']... (total valid files: 53)
Loading Combined Inputs...
Total files in final_generated_input: 1500
Loaded 1500 valid images from final_generated_input: ['aug_0000.png', 'aug_0001.png', 'aug_0002.pn

ValueError: No gradients provided for any variable.